# Causal Transformer: Entrenamiento y Evaluación

In [1]:
import os
import random
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import optuna
import time

os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from src.eval.losses import QLIKELoss
from src.eval.scorer import VolatilityScorer
from src.models.causal_transformer_wrapper import CausalTransformerModel
from src.trainer.walk_forward import WalkForwardTrainer
from src.utils import trainer_toolkit as tt
from src.utils import ts_toolkit as tst
from src.utils import test_toolkit as ts

warnings.filterwarnings("ignore")

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

OUT_PATH = PROJECT_ROOT / "data" / "processed"
RESULTS_PATH = PROJECT_ROOT / "results" / "causal_transformer"
RESULTS_PATH.mkdir(parents=True, exist_ok=True)

df_train = pd.read_parquet(OUT_PATH / "brent_train.parquet")
y_train = df_train["log_returns"].values.astype("float64")
rs_train = df_train["rs_daily"].values.astype("float64")

df_test = pd.read_parquet(OUT_PATH / "brent_test.parquet")
y_test = df_test["log_returns"].values.astype("float64")
rs_test = df_test["rs_daily"].values.astype("float64")

assert not np.isnan(y_train).any() and not np.isnan(rs_train).any()
assert not np.isnan(y_test).any() and not np.isnan(rs_test).any()

print("=" * 70)
print(f"Train: {len(y_train)} obs  ({df_train.index[0].date()} -> {df_train.index[-1].date()})")
print(f"Test:  {len(y_test)} obs  ({df_test.index[0].date()} -> {df_test.index[-1].date()})")
print("=" * 70)

Train: 2171 obs  (2014-01-03 -> 2022-11-03)
Test:  953 obs  (2022-11-07 -> 2026-08-21)


:::{admonition} Resumen
:class: info
- El modelo *Causal Transformer* se evaluó mediante un esquema de ventana expansiva con reentrenamiento periódico para el pronóstico de la volatilidad diaria del crudo Brent.
- En la comparación entre entrenamiento y prueba, QLIKE ($\Delta_{\text{abs}} = -0.0613$) y HMSE ($\Delta_{\text{abs}} = +0.5890$) superaron marginalmente el umbral de tolerancia de 0.05, manteniendo el resto de las métricas en rangos acotados y con residuos de colas pesadas mejor descritos por una distribución $t$ de Student ($\nu = 6.2950$) que por la normal.
- En evaluación fuera de muestra (*walk-forward*, 133.77 minutos o 8026.46 s para 953 observaciones), no se observó autocorrelación lineal ni dependencia no lineal en dimensión 2, pero persistió un efecto ARCH significativo en el rezago 20 ($p_{\text{adj}} = .024$).
- En el periodo de prueba y el episodio de volatilidad de 2026, el modelo siguió la dinámica general con ligera tendencia superior en calma relativa pero subestimó el pico principal de marzo, ubicándose a la par de referencias como TCN o GARCH aunque con limitaciones ante choques externos abruptos y un tiempo de cómputo por optimizar.
:::

## Introducción

El *Causal Transformer* (`CausalTransformerModel`) es una arquitectura de atención secuencial diseñada para capturar dependencias temporales no lineales a lo largo de una ventana histórica de longitud $L$ (`look_back`), garantizando estrictamente que la predicción en cada instante dependa únicamente de observaciones pasadas y presentes. A diferencia de las arquitecturas recurrentes, que propagan el estado oculto de forma secuencial paso a paso, el mecanismo de autoatención calcula relaciones entre todas las posiciones de la ventana en paralelo, permitiendo modelar interacciones entre rezagos distantes sin recurrencia ni dilataciones fijas.

En series financieras, esta formulación permite aprender patrones temporales no lineales entre retornos históricos estandarizados y la varianza futura del crudo Brent, evitando imponer una ecuación paramétrica rígida para la dinámica de la volatilidad condicional.

El objetivo de esta sección es presentar, entrenar y evaluar el modelo `CausalTransformerModel` para pronosticar la varianza realizada de Rogers-Satchell ($RS$) de los futuros del crudo Brent (`BZ=F`) bajo el protocolo del *benchmark*. Se analiza el efecto de la longitud de ventana (`look_back`), la dimensión del espacio latente (`d_model`), el número de cabezas de atención (`nhead`), la profundidad del *encoder* (`num_layers`), el ancho de la red *feed-forward* (`dim_feedforward`), la función de activación y la regularización (`dropout` y `weight_decay`) sobre el desempeño predictivo, manteniendo las mismas condiciones de información, pérdida **QLIKE** y evaluación *walk-forward* que el resto de modelos del proyecto.

### Arquitectura

La arquitectura de `CausalTransformerNetwork` se estructura sobre una proyección lineal de entrada, codificación posicional aprendible, un *encoder* de Transformer con máscara causal superior (`norm_first=True`), normalización final sobre el último paso temporal y una cabeza de salida positiva.

Dado un tensor de entrada $\mathbf{X}_t \in \mathbb{R}^{L \times F}$ (donde $L$ es `look_back` y $F$ es `in_features`, con $F = 1$ para la serie univariada de retornos estandarizados), la proyección lineal y la suma de la codificación posicional aprendible $\mathbf{P} \in \mathbb{R}^{1 \times L \times d_{\text{model}}}$ se expresan como:

$$
\mathbf{H}^{(0)} = \mathbf{X}_t \mathbf{W}_{\text{in}} + \mathbf{b}_{\text{in}} + \mathbf{P},
$$

donde $\mathbf{W}_{\text{in}} \in \mathbb{R}^{F \times d_{\text{model}}}$ y $\mathbf{b}_{\text{in}} \in \mathbb{R}^{d_{\text{model}}}$.

Para evitar cualquier fuga de información hacia adelante (*look-ahead bias*) dentro de la ventana de longitud $L$, se aplica una máscara causal $\mathbf{M} \in \mathbb{R}^{L \times L}$ definida mediante el triángulo superior estricto:

$$
M_{i,j} =
\begin{cases}
0, & \text{si } j \le i, \\
-\infty, & \text{si } j > i.
\end{cases}
$$

En cada capa $l \in \{1, \dots, \text{num\_layers}\}$ del *encoder* con pre-normalización (`norm_first=True`), el mecanismo de atención causal multi-cabeza (*Multi-Head Self-Attention*, MHSA) y el bloque *feed-forward* por posición (FFN) operan residualmente:

$$
\widetilde{\mathbf{H}}^{(l)} = \mathbf{H}^{(l-1)} + \text{MHSA}\left(\text{LayerNorm}\left(\mathbf{H}^{(l-1)}\right), \mathbf{M}\right),
$$

$$
\mathbf{H}^{(l)} = \widetilde{\mathbf{H}}^{(l)} + \text{FFN}\left(\text{LayerNorm}\left(\widetilde{\mathbf{H}}^{(l)}\right)\right),
$$

donde la red *feed-forward* expande la representación hacia `dim_feedforward` (por defecto $2 \cdot d_{\text{model}}$) con *Dropout* y activación seleccionable (`relu` o `gelu` en `_ACTIVATIONS`).

Una vez procesada la secuencia por las capas del *encoder*, se extrae la representación correspondiente al último instante de la ventana, $\mathbf{h}_L^{\text{num\_layers}} \in \mathbb{R}^{d_{\text{model}}}$, se aplica una capa `LayerNorm` final y se proyecta linealmente a un escalar:

$$
\mathbf{z}_L = \text{LayerNorm}\left(\mathbf{h}_L^{\text{num\_layers}}\right), \qquad y_t = \mathbf{w}_{\text{fc}}^{\top}\mathbf{z}_L + b_{\text{fc}}.
$$

Para garantizar que la varianza pronosticada sea estrictamente positiva y numéricamente estable, $y_t$ atraviesa la función *Softplus* sumada a un piso $\varepsilon = 10^{-8}$:

$$
\widehat{RS}_t = \text{Softplus}(y_t) + \varepsilon = \ln\left(1 + e^{y_t}\right) + \varepsilon,
$$

donde $\boldsymbol{\theta}$ reúne todos los parámetros entrenables de la proyección inicial, las incrustaciones posicionales, las capas del *encoder*, las normalizaciones y la capa de salida.

Siguiendo el protocolo del *benchmark*, los retornos logarítmicos diarios $r_t$ se estandarizan mediante transformación Z-score ajustada exclusivamente sobre el segmento de entrenamiento de cada *fold* para evitar fuga de información, mientras que la varianza realizada de Rogers-Satchell se escala mediante el factor de objetivo (`target_scale`). El modelo opera bajo un esquema de predicción de un paso adelante (*one-step-ahead forecasting*) y hereda de `BasePyTorchVolatilityModel` la integración con el *walk-forward* (`retrain_freq=1`), optimización con Adam, *weight decay*, *Dropout* y parada temprana (`patience=20`, `max_epochs=200`), minimizando la pérdida asimétrica **QLIKE**:

$$
\boldsymbol{\theta}^{*} = \arg\min_{\boldsymbol{\theta}} \frac{1}{N} \sum_{i=1}^{N} \mathcal{L}_{\text{QLIKE}}\left(RS_{i}, \widehat{RS}_{i}\right),
$$

donde $N$ es el número de ventanas de entrenamiento y $RS_i$ denota la varianza de Rogers-Satchell objetivo. La implementación se desarrolla en `PyTorch` (`2.13.0`) con diferenciación automática.

## Entrenamiento

Para la selección de hiperparámetros del modelo `CausalTransformerModel`, se mantuvo el mismo protocolo experimental (*Walk-Forward*) y el mismo presupuesto computacional que en los modelos paramétricos, recurrentes y convolucionales. A diferencia del GARCH y el EGARCH, la búsqueda de Optuna exploró la configuración de la atención causal y sus parámetros de regularización.

Para adaptar la serie temporal al formato requerido por la red, se construyó una matriz de rezagos (*lag matrix*) para cada ventana del esquema *Walk-Forward* (`n_splits=5`, con `warmup` ajustado según la ejecución y `embargo=1`). Siguiendo los principios de validación cruzada para series temporales (`timeseriescv`), cada fila de esta matriz contenía una secuencia histórica de longitud `look_back`, la cual se utilizó como vector de entrada para predecir el paso inmediatamente posterior. Este enfoque garantizó la integridad temporal de los datos y evitó cualquier tipo de fuga de información futura (*look-ahead bias*) durante el entrenamiento y el reentrenamiento (`retrain_freq=1`).

El espacio de búsqueda incluyó la longitud de la ventana de entrada (`look_back`), la dimensión oculta (`d_model`), el número de cabezas de atención (`nhead`), el número de capas del *encoder* (`num_layers`), la función de activación (`activation`), la tasa de abandono (`dropout`) y la tasa de aprendizaje (`learning_rate`). Para mantener la consistencia dimensional de la red *feed-forward*, `dim_feedforward` se definió como el doble de `d_model`, y para mitigar el sobreajuste se incorporó también la penalización $\ell_2$ (`weight_decay`).

El entrenamiento se configuró con un máximo de 200 épocas y una paciencia de 20 para la parada temprana interna de la red, bajo un esquema de búsqueda con `n_trials=50` y `early_stop_rounds=15`. El desempeño se siguió evaluando mediante la pérdida QLIKE sobre los conjuntos de validación.

In [ ]:
def causal_transformer_search_space(trial: optuna.Trial) -> dict:
    d_model = trial.suggest_categorical("d_model", [32, 64])
    nhead = trial.suggest_categorical("nhead", [2, 4])
    return {
        "look_back": trial.suggest_categorical("look_back", [7, 14, 21, 28]),
        "d_model": d_model,
        "nhead": nhead,
        "num_layers": trial.suggest_int("num_layers", 1, 2),
        "dim_feedforward": d_model * 2,
        "dropout": trial.suggest_float("dropout", 0.0, 0.3),
        "activation": trial.suggest_categorical("activation", ["relu", "gelu"]),
        "learning_rate": trial.suggest_float("learning_rate", 1e-4, 1e-2, log=True),
        "weight_decay": trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True),
        "max_epochs": 200,
        "patience": 20,
        "retrain_freq": 1,
        "seed": 42,
    }


exec_id = 2
current_warmup = 1200 + (exec_id * 10)

exec_dir = RESULTS_PATH / f"exec_{exec_id}"
exec_dir.mkdir(parents=True, exist_ok=True)

trainer = WalkForwardTrainer(
    model_class=CausalTransformerModel,
    scorer=VolatilityScorer(),
    output_dir=exec_dir,
    n_splits=5,
    warmup=current_warmup,
    embargo=1,
    n_trials=50,
    suggest_params_fn=causal_transformer_search_space,
    early_stop_rounds=15,
)

trainer.run(y_train, rs_train)

El algoritmo de búsqueda completó 49 ensayos válidos de 50 posibles (98 %), alcanzando un mejor valor de QLIKE con 0 ensayos podados y 0 fallidos. La ausencia de podas refleja que el codificador con atención causal procesa las ventanas en paralelo y estabiliza su entrenamiento mediante pre-normalización, conexiones residuales y parada temprana, lo que permite completar los ensayos sin descartes prematuros durante la búsqueda.

La búsqueda completa en Optuna tomó aproximadamente 23.50 horas, mientras que el ajuste y entrenamiento del modelo bajo el esquema *Walk-Forward* totalizó aproximadamente 11.41 minutos, manteniendo un bajo tiempo medio por partición. Este tiempo de ajuste evidencia el costo computacional de entrenar arquitecturas basadas en autoatención con Adam y retropropagación frente a modelos más ligeros, conservando la viabilidad operativa dentro del protocolo *Walk-Forward*.

In [3]:
resumen_data = []

exec_id = 2
model_tag = "CausalTransformer" 
study_name_override = None 
exec_dir = RESULTS_PATH / f"exec_{exec_id}"

tiempo_total_rep = 0.0
tiempo_promedio_fold = 0.0
num_folds = 0
total_trials, completos, podados, fallidos = 0, 0, 0, 0
mejor_qlike = "-"

csv_files = sorted(exec_dir.glob("*_results.csv"))
if csv_files:
    df_csv = pd.read_csv(csv_files[0])
    if "train_time_sec" in df_csv.columns:
        tiempo_total_rep = float(df_csv["train_time_sec"].sum())
        tiempo_promedio_fold = float(df_csv["train_time_sec"].mean())
    num_folds = len(df_csv)
else:
    print(f"No se encontró el CSV en {exec_dir}")

db_path = exec_dir / "optuna_study.db"
if db_path.exists():
    try:
        storage_url = f"sqlite:///{db_path.resolve()}"
        available_studies = optuna.get_all_study_names(storage=storage_url)

        if study_name_override and study_name_override in available_studies:
            target_study = study_name_override
        elif available_studies:
            target_study = available_studies[-1]
        else:
            target_study = f"{model_tag}_study"

        study = optuna.load_study(study_name=target_study, storage=storage_url)

        total_trials = len(study.trials)
        completos = sum(1 for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE)
        podados = sum(1 for t in study.trials if t.state == optuna.trial.TrialState.PRUNED)
        fallidos = sum(1 for t in study.trials if t.state == optuna.trial.TrialState.FAIL)

        if completos > 0 and study.best_value is not None:
            mejor_qlike = round(float(study.best_value), 4)

    except Exception as e:
        print(f"Error leyendo BD en {exec_dir}: {e}")
else:
    print(f"No se encontró la BD en {db_path}")

resumen_data.append({
    "Exec": exec_id,
    "Total Trials": total_trials,
    "Completos": completos,
    "Podados": podados,
    "Fallidos": fallidos,
    "Mejor QLIKE": mejor_qlike,
    "Tiempo Total (s)": round(tiempo_total_rep, 2),
    "Tiempo por Fold (s)": round(tiempo_promedio_fold, 3),
})

df_search_params = pd.DataFrame(resumen_data)
tt.display_styled_df(df=df_search_params, decimals=4, align_left_first=False)

Exec,Total Trials,Completos,Podados,Fallidos,Mejor QLIKE,Tiempo Total (s),Tiempo por Fold (s)
2,50,49,0,0,0.4921,684.7900,2.7720


El ranking de las cinco mejores configuraciones mostró un consenso en el uso de ventanas cortas de 7 días y activación `relu`, lo que sugiere que el historial reciente de una semana resulta informativo para capturar dinámicas de corto plazo en la volatilidad del Brent. En cuanto a la estructura del *encoder*, las configuraciones más destacadas se concentraron en 2 cabezas de atención y profundidades de 1 y 2 capas. Al carecer de un estado oculto recurrente como las redes secuenciales tradicionales, la atención causal sobre la ventana de entrada permite ponderar directamente las relaciones entre pasos temporales dentro del contexto seleccionado sin recurrencia paso a paso.

In [4]:
exec_dir = RESULTS_PATH / "exec_2"

csv_path = exec_dir / "CausalTransformer_results.csv"
if not csv_path.exists():
    fallback_csvs = sorted(exec_dir.glob("*_results.csv"))
    if not fallback_csvs:
        raise FileNotFoundError(f"No se encontró archivo de resultados en {exec_dir}")
    csv_path = fallback_csvs[0]

df_ct = pd.read_csv(csv_path)

df_ct["lr"] = df_ct["learning_rate"].apply(lambda x: f"{x:.4f}")
df_ct["wd"] = df_ct["weight_decay"].apply(lambda x: f"{x:.1e}")
df_ct["drop_fmt"] = df_ct["dropout"].apply(lambda x: f"{x:.3f}")

cols_agrupar = [
    "look_back",
    "d_model",
    "nhead",
    "num_layers",
    "dim_feedforward",
    "drop_fmt",
    "activation",
    "lr",
    "wd",
]

df_agg = (
    df_ct.groupby(cols_agrupar, as_index=False)["qlike"]
    .agg(mean="mean", std="std")
)
df_agg["std"] = df_agg["std"].fillna(0.0)
df_agg["QLIKE"] = df_agg.apply(
    lambda row: f"{row['mean']:.3f} ± {row['std']:.3f}", axis=1
)

df_top5 = df_agg.sort_values(by="mean", ascending=True).head(5).rename(
    columns={
        "look_back": "Lookback",
        "d_model": "d_model",
        "nhead": "Heads",
        "num_layers": "Capas",
        "dim_feedforward": "FFN",
        "drop_fmt": "Dropout",
        "activation": "Act",
        "lr": "LR",
        "wd": "Weight Decay",
    }
)

cols_finales = [
    "Lookback",
    "d_model",
    "Heads",
    "Capas",
    "FFN",
    "Dropout",
    "Act",
    "LR",
    "Weight Decay",
    "QLIKE",
]

df_final = df_top5[cols_finales]
tt.display_styled_df(df=df_final, align_left_first=False, decimals=3)

Lookback,d_model,Heads,Capas,FFN,Dropout,Act,LR,Weight Decay,QLIKE
7,32,2,2,64,0.214,relu,0.0071,6.0e-04,0.492 ± 0.210
7,64,2,1,128,0.246,relu,0.0081,7.4e-04,0.500 ± 0.213
7,64,2,1,128,0.202,relu,0.0077,5.2e-04,0.500 ± 0.243
7,64,2,1,128,0.271,relu,0.0065,5.2e-04,0.506 ± 0.230
7,32,2,2,64,0.097,relu,0.0095,2.4e-04,0.507 ± 0.234


Entre las dos primeras configuraciones, la diferencia de QLIKE es reducida frente a la dispersión observada entre *folds*, lo que hace que ambas sean cercanas en términos de validación. La elección de la primera configuración responde tanto a su menor pérdida media de validación como a un criterio de parsimonia al emplear un espacio latente más compacto (`d_model = 32` y `dim_feedforward = 64`), reduciendo la complejidad y la carga paramétrica del modelo sin sacrificar capacidad predictiva.

In [8]:
best_ct_model = CausalTransformerModel(
    look_back=7,
    d_model=32,
    nhead=2,
    num_layers=2,
    dim_feedforward=64,
    dropout=0.214,
    activation="relu",
    learning_rate=0.0071,
    weight_decay=6.0e-4,
    max_epochs=200,
    patience=20,
    retrain_freq=1,
    seed=42,
)

start_time = time.time()
best_ct_model.fit(y_train, rs_train)
end_time = time.time()

elapsed_time = end_time - start_time

El entrenamiento del modelo sobre toda la muestra tuvo una duración de 15.96 segundos (15.9568 s). En la gráfica de aprendizaje, tras un pico pronunciado de validación en la segunda época y oscilaciones iniciales en ambas trayectorias, el QLIKE descendió rápidamente hacia valores bajos y se estabilizó en torno a la época 9, donde se seleccionaron los mejores pesos y el entrenamiento continuó hasta cerca de la época 29.

A diferencia de la TCN, donde la validación solía mantenerse por debajo de entrenamiento, en este caso la curva de validación superó con rapidez y de forma persistente a la de entrenamiento a partir de la selección de mejores pesos. Este comportamiento sugirió una mayor tendencia del modelo a ajustar con más fuerza la muestra de entrenamiento frente a la validación, acorde con la flexibilidad de la atención causal, aunque la regularización y la parada temprana evitaron una divergencia explosiva en la pérdida en el conjunto validación. La meseta posterior indicó que el modelo capturó el nivel base de la varianza condicional, pero con margen acotado para refinar picos extremos sin perder generalización.

In [10]:
import pandas as pd

history_csv_path = exec_dir / "causal_transformer_training_history.csv"
history_df = pd.read_csv(history_csv_path)

if "epoch" in history_df.columns:
    history_df = history_df.drop(columns=["epoch"])

history_from_csv = {
    col: history_df[col].dropna().tolist()
    for col in history_df.columns
}

n_epochs = len(history_df)
x_max = max(n_epochs + 1, 2)
x_step = max(1, round(x_max / 11))

tt.plot_learning_curve(
    history=history_from_csv,
    title="Curva de Aprendizaje",
    ylab="Quasi-Likelihood (QLIKE)",
    xlab_pad=14,
    ylab_pad=14,
    line_width=2.0,
    title_size=14,
    label_size=14,
    tick_size=12,
    legend_size=14,
    ylim=(0, 6),
    xlim=(0, x_max),
    y_step=0.5,
    x_step=x_step,
    legend_bbox=(0.95, 0.95),
)

Para evaluar el ajuste dentro de la muestra de entrenamiento, a diferencia de los modelos paramétricos, no fue posible extraer los residuos directamente del objeto del modelo. En su lugar, se realizó un pase hacia adelante (*forward pass*) sobre la serie completa de entrenamiento: primero se transformó `y_train` con el `StandardScaler` ajustado durante `fit` y se escaló el objetivo de varianza, luego se construyó la matriz de rezagos con `TimeSeriesDataset` usando ventanas de 7 días (`look_back = 7`) y finalmente se evaluó en modo `eval()` sin gradientes ni regularización. Las varianzas estimadas se reescalaron a su unidad original y se alinearon con los datos reales descartando las observaciones iniciales correspondientes a la ventana.

In [ ]:
best_ct_model.net.eval()
look_back = best_ct_model.context_length
target_scale = getattr(best_ct_model, "target_scale", 1.0)

y_train_scaled = best_ct_model.scaler.transform(
    np.asarray(y_train).reshape(-1, 1)
).flatten()
rs_train_scaled = np.asarray(rs_train) * target_scale

train_eval_dataset = TimeSeriesDataset(
    y_train_scaled,
    rs_train_scaled,
    look_back,
)
train_eval_loader = DataLoader(
    train_eval_dataset,
    batch_size=getattr(best_ct_model, "batch_size", 128),
    shuffle=False,
)

preds_train_list = []
with torch.no_grad():
    for x_batch, _ in train_eval_loader:
        x_batch = x_batch.to(best_ct_model.device)
        preds_batch = best_ct_model.net(x_batch).squeeze(-1).detach().cpu().numpy()
        preds_train_list.append(preds_batch)

preds_train_var = np.concatenate(preds_train_list) / target_scale
n_preds = len(preds_train_var)

start_idx = len(y_train) - n_preds
dates_aligned = df_train.index[start_idx : start_idx + n_preds]
y_train_aligned = np.asarray(y_train)[start_idx : start_idx + n_preds]
rs_train_aligned = np.asarray(rs_train)[start_idx : start_idx + n_preds]

df_train_preds = pd.DataFrame(
    {
        "y_true": y_train_aligned,
        "rs_true": rs_train_aligned,
        "y_pred": preds_train_var,
    },
    index=dates_aligned,
)

train_output_path = exec_dir / "causal_transformer_train_predictions.csv"
df_train_preds.to_csv(train_output_path, index=True)


A partir de las varianzas condicionales predichas $\left(\hat{\sigma}_t^2\right)$ y de los retornos observados $\left(y_t\right)$, se evaluó el ajuste del modelo sobre el conjunto de entrenamiento mediante los residuos estandarizados:

$$
z_t = \frac{y_t}{\sqrt{\hat{\sigma}_t^2}}
$$

El diagnóstico sobre los residuos estandarizados mostró ausencia de autocorrelación lineal significativa tras el ajuste de Bonferroni en las pruebas de Ljung-Box para los rezagos 10, 20 y 30 ($p_{\text{adj}} \ge .084$), así como ausencia de dependencia no lineal en la prueba BDS en dimensión 2 ($p = .781$). Las pruebas ARCH-LM rechazaron la hipótesis nula de homocedasticidad condicional en los tres rezagos evaluados ($p_{\text{adj}} < .001$), lo que evidenció que persistió agrupamiento residual de volatilidad dentro de la muestra de entrenamiento. Asimismo, las pruebas de Jarque-Bera y Lilliefors rechazaron la normalidad de los residuos ($p < .001$), reflejando colas pesadas características de los retornos del crudo Brent.

In [4]:
import numpy as np
import pandas as pd

train_output_path = exec_dir / "causal_transformer_train_predictions.csv"

df_train_preds = pd.read_csv(train_output_path)
rs_train_aligned = df_train_preds["rs_true"].to_numpy()
preds_train_var = df_train_preds["y_pred"].to_numpy()

n_preds = len(preds_train_var)
y_arr = np.asarray(y_train).reshape(-1)
y_train_aligned = y_arr[-n_preds:]

std_residuals_train = y_train_aligned / np.sqrt(np.maximum(preds_train_var, 1e-12))
std_residuals_train = std_residuals_train[np.isfinite(std_residuals_train)]

df_diag = tt.run_model_diagnostics(
    std_residuals=std_residuals_train,
    lags=(10, 20, 30),
    p_adjust="bonferroni",
    alpha=0.05,
    english=False,
    column_titles=(
        "Prueba Estadística",
        "Estadístico",
        "p-valor",
        "p-adj",
        "Conclusión (α = 0.05)",
    ),
)

df_display = df_diag.copy()

df_display["Estadístico"] = df_display["Estadístico"].apply(
    lambda x: f"{x:.4f}" if pd.notna(x) else "--"
)


def _format_p(x):
    if pd.isna(x):
        return "--"
    if x < 0.001:
        return "< .001"
    formatted = f"{x:.3f}"
    return formatted[1:] if formatted.startswith("0.") else formatted


df_display["p-valor"] = df_display["p-valor"].apply(_format_p)
df_display["p-adj"] = df_display["p-adj"].apply(_format_p)

tt.display_styled_df(df=df_display, p_value_cols=[])

Prueba Estadística,Estadístico,p-valor,p-adj,Conclusión (α = 0.05)
Ljung-Box (Lag 10),10.6411,.386,1.000,Sin Autocorrelación
Ljung-Box (Lag 20),33.7377,.028,.084,Sin Autocorrelación
Ljung-Box (Lag 30),45.8004,.032,.097,Sin Autocorrelación
ARCH-LM (Lag 10),39.4294,< .001,< .001,Efectos ARCH Significativos
ARCH-LM (Lag 20),58.1490,< .001,< .001,Efectos ARCH Significativos
ARCH-LM (Lag 30),67.1109,< .001,< .001,Efectos ARCH Significativos
Jarque-Bera,1241.5282,< .001,--,Residuos No Normales
Lilliefors,0.0649,< .001,--,Residuos No Normales
BDS (Dim 2),-0.2784,.781,--,Sin Estructura No Lineal


Los residuos estandarizados del modelo mostraron un ajuste a una distribución $t$ de Student ($\nu = 3.9281$). En los cuantiles extremos de ambas colas se observaron desviaciones respecto a la referencia teórica, con los cuantiles superiores empíricos situándose por debajo de los teóricos. Estas discrepancias en las colas implicaron que la aproximación $t$ de Student sobrestimó ligeramente la magnitud de los eventos extremos en comparación con la cola empírica de los residuos, reflejando un ajuste más preciso en la región central que en los valores atípicos más severos. Pese a ello, las pruebas formales de bondad de ajuste (Kolmogorov-Smirnov: $p = .4105$; Cramér-von Mises: $p = .3213$) no rechazaron la hipótesis nula de ajuste a la distribución $t$.

In [6]:
_, _ = tst.plot_distribution_fit(
    pd.Series(std_residuals_train),
    dist="t",
    titles=("Distribución de Residuos", "Gráfico Q-Q (t-Student)"),
    xlab="Residuo estandarizado",
    ylab_left="Densidad",
    xlab_right="Cuantiles teóricos",
    ylab_right="Cuantiles muestrales",
    ylim_left=(0, 0.7),
    legend_loc="upper left"
)

La prueba de razón de verosimilitud (LRT) rechazó la hipótesis nula de igualdad de ajuste frente a la distribución normal (estadístico $= 254.93$, $p < .001$). Asimismo, los criterios de información mostraron una reducción de $\Delta\text{AIC} = 252.93$ y $\Delta\text{BIC} = 247.25$ a favor de la distribución $t$ de Student. Estos resultados apoyaron que la distribución $t$ capturara con mayor precisión las colas pesadas de los residuos estandarizados del modelo en la muestra de entrenamiento en comparación con el ajuste Gaussiano.

In [7]:
_= tst.distribution_fit_table(
    pd.Series(std_residuals_train),
    column_titles=(
        "Distribución", 
        "Log-Likelihood", 
        "AIC", 
        "BIC", 
        "Estadístico LRT", 
        "p-valor LRT"
    )
)

Distribución,Log-Likelihood,AIC,BIC,Estadístico LRT,p-valor LRT
Normal,"-3,250.17","6,504.35","6,515.71",-,-
t-Student,"-3,122.71","6,251.42","6,268.46",254.93,< .001


A diferencia de las arquitecturas evaluadas previamente, el *Causal Transformer* reaccionó con mayor rapidez a los episodios de alta volatilidad e incluso llegó a sobreestimar picos puntuales en periodos de choque, como se aprecia en el acercamiento a la crisis de 2020. Este comportamiento se puede deber que la atención causal asigna pesos elevados a los rezagos más recientes y recientes picos de varianza dentro de la ventana de entrada, lo que facilita un ajuste estrecho sobre los datos de entrenamiento y aumenta el riesgo de sobreestimar saltos bruscos. Con el fin de obtener una evaluación más completa de su capacidad de generalización, a continuación se analizó su desempeño en el conjunto de prueba.

In [10]:
look_back = best_ct_model.context_length
n_preds = len(preds_train_var)
start_idx = len(y_train) - n_preds

dates_aligned = df_train.index[start_idx : start_idx + n_preds]
rs_volatilidad_aligned = np.sqrt(np.maximum(np.asarray(rs_train)[start_idx : start_idx + n_preds], 0.0))
pred_volatilidad = np.sqrt(np.maximum(preds_train_var, 0.0))

mask_crisis = (dates_aligned >= "2020-01-01") & (dates_aligned <= "2020-06-30")

ts.plot_volatility_fit(
    dates=dates_aligned,
    rs_variance=rs_volatilidad_aligned,
    pred_variance=pred_volatilidad,
    title="Ajuste dentro de la Muestra de Entrenamiento (Causal Transformer)",
    ylab="Volatilidad",
    label_realized=r"Volatilidad Realizada $\left(\sigma_R\right)$",
    label_pred=r"Volatilidad Causal Transformer $\left(\hat{\sigma}_t\right)$",
    ylim=(-0.01, 0.7),
    legend_bbox=(0.01, 0.98),
    zoom_slice=mask_crisis,
    zoom_pad_frac=0.04,
    zoom_title="Zoom: Crisis COVID-19",
    zoom_bbox=(0.18, 0.35, 0.40, 0.40),
    title_size=14,
    label_size=14,
    tick_size=12,
    legend_size=14,
)

## Evaluación

La evaluación del rendimiento de un modelo de volatilidad requiere contrastar su ajuste sobre los datos observados con su capacidad predictiva en observaciones no vistas. Mientras que el conjunto de entrenamiento permite verificar la convergencia del algoritmo sobre la dinámica histórica, la evaluación fuera de muestra determina el desempeño del modelo ante nuevos regímenes de mercado.

Para diagnosticar posibles problemas de generalización, se aplicaron los criterios pre-registrados del protocolo experimental a partir de la diferencia absoluta entre las métricas de prueba y entrenamiento:

$$
\Delta_{\text{abs}} = M_{\text{test}} - M_{\text{train}}
$$

Se fijó un umbral de tolerancia de $|\Delta_{\text{abs}}| = 0.05$ para las métricas evaluadas, de modo que discrepancias por encima de este margen señalan cambios relevantes entre el ajuste en muestra y el desempeño fuera de muestra.

En el modelado de volatilidad con pérdida QLIKE, esta lectura debe matizarse por la asimetría de la función de pérdida, que castiga con mayor severidad la subestimación de la varianza que su sobreestimación. Un modelo con sesgo alcista o defensivo puede registrar un QLIKE menor en prueba que en entrenamiento sin que ello implique una mejor capacidad estructural, sino una coincidencia favorable entre el nivel predicho y el régimen de volatilidad del periodo de prueba. Por ello, el análisis de $\Delta_{\text{abs}}$ se complementa con la inspección de las trayectorias predichas y el diagnóstico de residuos estandarizados.

Las predicciones fuera de muestra se generaron mediante el método `forecast` del modelo, aplicando una ventana expansiva con reentrenamiento periódico (`retrain_freq = 1`) sin fuga de información y devolviendo las varianzas condicionales en su escala original para compararlas con la volatilidad realizada.

In [ ]:
start_time = time.time()
preds_test = best_ct_model.forecast(y_seq=y_test, rs_seq=rs_test)
preds_time = time.time() - start_time

preds_test_arr = np.asarray(preds_test).flatten()
rs_test_arr = np.asarray(rs_test).flatten()
y_test_arr = np.asarray(y_test).flatten()

n_test = len(preds_test_arr)
dates_test_aligned = df_test.index[-n_test:]
rs_test_aligned = rs_test_arr[-n_test:]
y_test_aligned = y_test_arr[-n_test:]

df_preds = pd.DataFrame(
    {
        "date": dates_test_aligned,
        "y_true": y_test_aligned,
        "rs_true": rs_test_aligned,
        "y_pred": preds_test_arr,
    }
)

output_path = results_dir / "causal_transformer_test_predictions.csv"
df_preds.to_csv(output_path, index=False)

La generación de pronósticos fuera de muestra mediante el esquema de ventana expansiva y reentrenamiento periódico tuvo una duración total de 133.77 minutos (8026.46 s) para las 953 observaciones del conjunto de prueba, lo que equivale a un tiempo promedio de 0.14 minutos (unos 8.42 segundos) por observación. Este tiempo es sustancialmente superior al de los modelos paramétricos, cuyo cómputo toma milisegundos, reflejando el costo computacional de actualizar y reajustar los pesos del Causal Transformer a lo largo de la ventana de evaluación.

El diagnóstico de los residuos estandarizados fuera de muestra mostró ausencia de autocorrelación lineal significativa tras el ajuste de Bonferroni en las pruebas de Ljung-Box para los rezagos 10, 20 y 30 ($p_{\text{adj}} \ge .063$), así como ausencia de dependencia no lineal en la prueba BDS en dimensión 2 ($p = .096$). En las pruebas ARCH-LM, el rezago 20 presentó efectos ARCH significativos tras el ajuste de Bonferroni ($p_{\text{adj}} = .024$), mientras que en los rezagos 10 y 30 no se rechazó la hipótesis nula de homocedasticidad condicional ($p_{\text{adj}} = .202$ y $p_{\text{adj}} = .068$, respectivamente). Asimismo, las pruebas de Jarque-Bera ($p < .001$) y Lilliefors ($p = .002$) rechazaron la normalidad de los residuos.

In [6]:
preds_path = exec_dir / "causal_transformer_test_predictions.csv"
df_preds_test = pd.read_csv(preds_path)
preds_test_var = df_preds_test["y_pred"].values

std_residuals_oos = y_test / np.sqrt(np.maximum(preds_test_var, 1e-12))
std_residuals_oos = std_residuals_oos[np.isfinite(std_residuals_oos)]

df_diag = tt.run_model_diagnostics(
    std_residuals=std_residuals_oos,
    lags=(10, 20, 30),
    p_adjust="bonferroni",
    alpha=0.05,
    english=False,
    column_titles=(
        "Prueba Estadística",
        "Estadístico",
        "p-valor",
        "p-adj",
        "Conclusión (α = 0.05)",
    ),
)

df_display = df_diag.copy()

df_display["Estadístico"] = df_display["Estadístico"].apply(
    lambda x: f"{x:.4f}" if pd.notna(x) else "--"
)


def _format_p(x):
    if pd.isna(x):
        return "--"
    if x < 0.001:
        return "< .001"
    formatted = f"{x:.3f}"
    return formatted[1:] if formatted.startswith("0.") else formatted


df_display["p-valor"] = df_display["p-valor"].apply(_format_p)
df_display["p-adj"] = df_display["p-adj"].apply(_format_p)

tt.display_styled_df(df=df_display, p_value_cols=[])

Prueba Estadística,Estadístico,p-valor,p-adj,Conclusión (α = 0.05)
Ljung-Box (Lag 10),11.7932,.299,.897,Sin Autocorrelación
Ljung-Box (Lag 20),34.8187,.021,.063,Sin Autocorrelación
Ljung-Box (Lag 30),40.5283,.095,.285,Sin Autocorrelación
ARCH-LM (Lag 10),17.3331,.067,.202,Sin Efectos ARCH
ARCH-LM (Lag 20),38.2952,.008,.024,Efectos ARCH Significativos
ARCH-LM (Lag 30),47.4159,.023,.068,Sin Efectos ARCH
Jarque-Bera,142.9770,< .001,--,Residuos No Normales
Lilliefors,0.0396,.002,--,Residuos No Normales
BDS (Dim 2),-1.6650,.096,--,Sin Estructura No Lineal


La persistencia de un efecto ARCH remanente en el rezago 20 indica que la varianza condicional estimada no absorbe por completo la agrupación de volatilidad de mediano-corto plazo en la muestra de prueba. Dado que el objetivo del modelo es capturar la dinámica temporal de la volatilidad y estandarizar adecuadamente los choques, esta heterocedasticidad residual señala una reacción parcial o rezagada ante cambios de régimen antes de que el efecto se disipe en horizontes mayores.

La inspección visual de los residuos estandarizados fuera de muestra muestra un ajuste general adecuado a la distribución $t$ de Student con $\nu = 6.2950$ grados de libertad. Los cuantiles muestrales siguen de cerca la diagonal teórica a lo largo del centro y gran parte de los extremos, observándose únicamente leves desviaciones por debajo de la línea de referencia en los cuantiles inferiores (alrededor de $-4$ a $-2$) y en los cuantiles superiores más altos. Las pruebas formales de bondad de ajuste (Kolmogorov-Smirnov: $p = .6439$; Cramér-von Mises: $p = .7156$) no rechazaron la hipótesis nula de pertenencia a la distribución $t$.

In [7]:
_, _ = tst.plot_distribution_fit(
    pd.Series(std_residuals_oos),
    dist="t",
    titles=("Distribución de Residuos", "Gráfico Q-Q (t-Student)"),
    xlab="Residuo estandarizado",
    ylab_left="Densidad",
    xlab_right="Cuantiles teóricos",
    ylab_right="Cuantiles muestrales",
    ylim_left=(0, 0.7),
    legend_loc="upper left"
)

La prueba de razón de verosimilitud (LRT) rechazó la hipótesis nula de igualdad de ajuste frente a la distribución normal en los residuos estandarizados en el conjunto de prueba ($p < .001$). Los criterios de información mostraron una reducción de $\Delta\text{AIC} = 45.62$ y $\Delta\text{BIC} = 40.76$ a favor de la distribución $t$ de Student. Estos resultados apoyaron que la distribución $t$ capturara con mayor precisión las colas pesadas de los residuos estandarizados fuera de muestra en comparación con el ajuste Gaussiano.

In [8]:
_= tst.distribution_fit_table(
    pd.Series(std_residuals_oos),
    column_titles=(
        "Distribución", 
        "Log-Likelihood", 
        "AIC", 
        "BIC", 
        "Estadístico LRT", 
        "p-valor LRT"
    )
)

Distribución,Log-Likelihood,AIC,BIC,Estadístico LRT,p-valor LRT
Normal,"-1,399.15","2,802.31","2,812.03",-,-
t-Student,"-1,375.35","2,756.69","2,771.27",47.61,< .001


En la comparación entre entrenamiento y prueba, el modelo excedió el umbral de tolerancia de $|\Delta_{\text{abs}}| = 0.05$ en QLIKE ($\Delta_{\text{abs}} = -0.0613$) y en HMSE ($\Delta_{\text{abs}} = +0.5890$), mientras que el resto de las métricas (CRPS, pérdidas *pinball*, MSE, RMSE, MAE y HMAE con $\Delta_{\text{abs}} = -0.0304$) permanecieron dentro del margen de tolerancia y en magnitudes bajas. Que estas diferencias superen de forma marginal el umbral no implica necesariamente sobreajuste del modelo: este orden en las métricas puede estar asociado a cambios en el régimen de volatilidad entre muestras o a la transición entre el ajuste *in-sample* y el pronóstico secuencial *out-of-sample*. En particular, la reducción de QLIKE en prueba responde a la asimetría de la función de pérdida y al régimen de volatilidad del periodo de evaluación, mientras que el aumento de HMSE refleja el mayor peso relativo de las discrepancias en regímenes de baja varianza o choques puntuales.

In [9]:
look_back = 7
rs_train_aligned = rs_train[look_back:]

ts.display_train_test_comparison(
    rs_train=rs_train_aligned,
    preds_train=preds_train_var,
    rs_test=rs_test,
    preds_test=preds_test_var,
    column_titles=(
        "Métrica",
        "Entrenamiento",
        "Prueba",
        "Diferencia",
    ),
)

Métrica,Entrenamiento,Prueba,Diferencia
QLIKE,0.4549,0.3936,-0.0613
CRPS,0.0128,0.0123,-0.0005
Pinball (τ=0.05),0.0028,0.0027,-0.0001
Pinball (τ=0.95),0.0015,0.0016,+0.0001
MSE,0.0000,0.0000,-0.0000
RMSE,0.0023,0.0021,-0.0002
MAE,0.0005,0.0004,-0.0000
HMSE,1.5579,2.1469,+0.5890
HMAE,0.7158,0.6853,-0.0304


La inspección visual de las predicciones fuera de muestra del *Causal Transformer* evidenció que el modelo capturó la dinámica general de la volatilidad realizada ($\sigma_R$) y tendió a ubicarse ligeramente por encima en los periodos de calma relativa. Sin embargo, ante el choque de volatilidad de 2026, subestimó la magnitud del pico máximo alcanzado en marzo, siguiendo el patrón observado en los modelos evaluados previamente pese a su ajuste cercano sobre el conjunto de entrenamiento. Este comportamiento ayudó a explicar el incremento de HMSE ($\Delta_{\text{abs}} = +0.5890$) y la persistencia de heterocedasticidad residual de mediano-corto plazo en el rezago 20.

In [10]:
df_preds_test["date"] = pd.to_datetime(df_preds_test["date"])

rs_test_vol = np.sqrt(np.maximum(df_preds_test["rs_true"], 0.0))
pred_test_vol = np.sqrt(np.maximum(df_preds_test["y_pred"], 0.0))

mask_crisis = (df_preds_test["date"] >= "2026-02-01") & (df_preds_test["date"] <= "2026-06-30")

ts.plot_volatility_fit(
    dates=df_preds_test["date"],
    rs_variance=rs_test_vol,
    pred_variance=pred_test_vol,
    title="Ajuste Fuera de la Muestra (Causal Transformer)",
    ylab="Volatilidad",
    label_realized=r"Volatilidad Realizada $\left(\sigma_R\right)$",
    label_pred=r"Volatilidad Causal Transformer $\left(\hat{\sigma}_t\right)$",
    ylim=(-0.01, 0.35),
    legend_bbox=(0.01, 0.98),
    zoom_slice=mask_crisis,
    zoom_pad_frac=0.04,
    zoom_title="Zoom: Pico de Volatilidad (2026)",
    zoom_bbox=(0.25, 0.35, 0.40, 0.40),
    title_size=14,
    label_size=14,
    tick_size=12,
    legend_size=14,
)

## Conclusiones

La implementación y evaluación del modelo *Causal Transformer* permitieron contrastar el desempeño de una arquitectura basada en atención causal para el pronóstico de la volatilidad diaria del crudo Brent. En la comparación entre entrenamiento y prueba, el modelo superó el umbral de tolerancia de $|\Delta_{\text{abs}}| = 0.05$ en QLIKE ($\Delta_{\text{abs}} = -0.0613$) y en HMSE ($\Delta_{\text{abs}} = +0.5890$), sin que ello implique necesariamente sobreajuste, dado que estas variaciones marginales pueden asociarse a cambios en el régimen de volatilidad o a la transición entre el ajuste *in-sample* y la evaluación secuencial *out-of-sample*, mientras que el resto de las métricas se mantuvieron en rangos acotados.

En el diagnóstico fuera de muestra, los residuos estandarizados no mostraron autocorrelación lineal significativa tras el ajuste de Bonferroni en las pruebas de Ljung-Box para los rezagos 10, 20 y 30 ni dependencia no lineal en la prueba BDS en dimensión 2. En las pruebas ARCH-LM, se detectó un efecto ARCH significativo en el rezago 20, pero no en los rezagos 10 y 30, señalando una absorción incompleta de la heterocedasticidad condicional ante cambios de régimen. Asimismo, se rechazó la normalidad en Jarque-Bera y Lilliefors, y los residuos presentaron colas pesadas mejor representadas por una distribución $t$ de Student ($\nu = 6.2950$) que por la normal, con pruebas de Kolmogorov-Smirnov y Cramér-von Mises favorables a este ajuste.

En términos de trayectoria y cómputo, el modelo capturó la dinámica general de la volatilidad pero subestimó el pico de marzo de 2026, y el esquema secuencial con reentrenamiento periódico requirió 133.77 minutos (8026.46 s) para las 953 observaciones de prueba (0.14 minutos o 8.42 segundos por observación en promedio). En conjunto, el *Causal Transformer* se situó a la par de referencias sólidas como la TCN o los modelos tipo GARCH, ofreciendo un ajuste competitivo entre los mejores enfoques estudiados hasta el momento, aunque mantiene limitaciones frente a choques externos abruptos y un costo computacional elevado que conviene optimizar.